[Open in Colab](https://colab.research.google.com/github/fcoliveira-utfpr/aquacrop_ml/blob/main/11b_assistente_ia.ipynb)

# Assistente de IA (DeepSeek + MCP) — aquacrop_ml

Versão com texto livre do `11_assistente_perguntas.ipynb`: em vez de escolher a consulta num menu,
você pergunta em português e o modelo da **DeepSeek** decide quais tools do servidor MCP
(`mcp_maiz/server.py`) chamar para responder — as mesmas tools que respondem sobre melhor data de
semeadura, produtividade prevista, risco climático/econômico, custos DERAL/CONAB e preços.

**Diferente do `11`, este notebook precisa de uma chave de API da DeepSeek e tem custo** (baixo — é
uma das APIs mais baratas do mercado). Se você só quer consultar sem gastar nada, use o
`11_assistente_perguntas.ipynb`.

**Antes de rodar**: crie uma chave em platform.deepseek.com → API Keys, depois salve como secret do
Colab (ícone de chave 🔑 na barra lateral esquerda) com o nome `DEEPSEEK_API_KEY` e acesso liberado
para este notebook — assim você não digita a chave toda vez e ela nunca fica escrita no notebook.

**Para usar**: `Arquivo > Salvar uma cópia no Drive` e rode as células em ordem.


## 1. Setup

Instala o SDK `mcp` (fixado abaixo da versão 2.0, que renomeou a API `FastMCP` usada pelo
`mcp_maiz/server.py`) e o SDK `openai`, usado aqui só para falar com a API da DeepSeek — ela é
compatível com o formato de function calling da OpenAI, então o mesmo cliente funciona trocando
`base_url`. Clona o repositório se estiver rodando no Colab.


In [ ]:
!pip install -q "mcp<2" openai

import os
import sys

RODANDO_NO_COLAB = "google.colab" in str(get_ipython())

if RODANDO_NO_COLAB:
    if not os.path.exists("aquacrop_ml"):
        !git clone -q https://github.com/fcoliveira-utfpr/aquacrop_ml.git
    if os.path.basename(os.getcwd()) != "aquacrop_ml":
        os.chdir("aquacrop_ml")

sys.path.insert(0, "mcp_maiz")
print("Setup OK — rodando no Colab" if RODANDO_NO_COLAB else "Setup OK — rodando localmente")


## 2. Chave da API (DeepSeek)

Lê o secret `DEEPSEEK_API_KEY` do Colab. Se não estiver rodando no Colab (ou o secret não existir),
pede a chave digitada — nesse caso ela fica só na memória desta sessão, nunca é salva no notebook.


In [ ]:
try:
    from google.colab import userdata
    DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
    print("Chave carregada do Colab Secrets.")
except Exception:
    import getpass
    DEEPSEEK_API_KEY = getpass.getpass("Chave da API DeepSeek: ")


## 3. Conectando ao servidor MCP

Sobe `mcp_maiz/server.py` como subprocesso (transporte stdio) e mantém a sessão aberta durante toda
a conversa — evita reconectar (e recarregar os CSVs/modelo) a cada pergunta.


In [ ]:
import sys
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

_exit_stack = AsyncExitStack()

server_params = StdioServerParameters(
    command=sys.executable,
    args=["mcp_maiz/server.py"],
)

read, write = await _exit_stack.enter_async_context(stdio_client(server_params))
session = await _exit_stack.enter_async_context(ClientSession(read, write))
await session.initialize()

tools_mcp = (await session.list_tools()).tools
print(f"{len(tools_mcp)} tools conectadas ao servidor mcp_maiz:")
for t in tools_mcp:
    resumo = (t.description or "").strip().splitlines()[0] if t.description else ""
    print(f"  - {t.name}: {resumo}")


## 4. Cliente DeepSeek e loop do agente

Converte o schema de cada tool MCP (`inputSchema`, já em JSON Schema) para o formato de function
calling da DeepSeek. `perguntar()` implementa o loop: manda a pergunta + histórico pro modelo,
executa as tools que ele pedir via MCP, devolve o resultado pro modelo, repete até ele responder sem
pedir mais tools.

Prioriza `structuredContent` no resultado da tool (já JSON válido, com os tipos numéricos corretos);
cai para o texto concatenado só quando a tool não devolve `structuredContent` — que é também o
formato usado pra mensagens de erro (ex. município não encontrado), então o modelo consegue ler o
erro e sugerir corrigir a pergunta.


In [ ]:
import json

from openai import OpenAI

deepseek = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

tools_openai = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description or "",
            "parameters": t.inputSchema,
        },
    }
    for t in tools_mcp
]

SYSTEM_PROMPT = """Você é o assistente do projeto aquacrop_ml, que simula produtividade de milho \
2ª safra em 50 municípios da Mesorregião Oeste do Paraná (balanço hídrico FAO + machine learning) \
e avalia risco climático (ISNA/ZARC) e viabilidade econômica (referência SEAB/DERAL) por data de \
semeadura. Responda sempre em português, de forma direta. Use as tools disponíveis para buscar os \
dados — nunca invente números. Municípios fora dos 50 do Oeste do PR só têm dados via \
prever_produtividade_customizada (mais lenta, roda a simulação na hora)."""


def _texto_resultado(resultado) -> str:
    if resultado.structuredContent is not None:
        return json.dumps(resultado.structuredContent, ensure_ascii=False)
    return "\n".join(bloco.text for bloco in resultado.content if bloco.type == "text")


async def perguntar(pergunta: str, historico: list[dict] | None = None) -> tuple[str, list[dict]]:
    mensagens = historico if historico is not None else [{"role": "system", "content": SYSTEM_PROMPT}]
    mensagens.append({"role": "user", "content": pergunta})

    while True:
        resposta = deepseek.chat.completions.create(
            model="deepseek-chat",
            messages=mensagens,
            tools=tools_openai,
            tool_choice="auto",
        )
        msg = resposta.choices[0].message
        mensagens.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            return msg.content, mensagens

        for chamada in msg.tool_calls:
            argumentos = json.loads(chamada.function.arguments or "{}")
            print(f"  [tool] {chamada.function.name}({argumentos})")
            resultado = await session.call_tool(chamada.function.name, argumentos)
            mensagens.append({
                "role": "tool",
                "tool_call_id": chamada.id,
                "content": _texto_resultado(resultado),
            })


## 5. Conversar

Loop de chat na própria célula — digite a pergunta e aperte Enter. Digite `sair` para encerrar.


In [ ]:
historico = None

print('Digite sua pergunta (ou "sair" para encerrar).\n')

while True:
    pergunta = input("Você: ").strip()
    if not pergunta:
        continue
    if pergunta.lower() in {"sair", "exit", "quit"}:
        break
    resposta, historico = await perguntar(pergunta, historico)
    print(f"\nAssistente: {resposta}\n")


## 6. Encerrar (opcional)

Fecha o subprocesso do servidor MCP. Rode antes de sair do notebook, ou só encerre a sessão do
Colab/kernel — o subprocesso morre junto de qualquer forma.


In [ ]:
await _exit_stack.aclose()
print("Sessão MCP encerrada.")
